### Middleware

They provide a way to more tightly control what happens inside an agent.

https://docs.langchain.com/oss/python/langchain/middleware/built-in

In [9]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

#### Summarization
https://docs.langchain.com/oss/python/langchain/middleware/built-in#summarization

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Message based summarization
agent = create_agent(
    model="gpt-4.1",
    # tools=[your_weather_tool, your_calulator_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4.1-mini",
            trigger=("messages", 10), # condition based on the message limit
            keep=("messages", 4),
        ),
    ],
)

In [15]:
# specify the session configuration
config = {"configurable": {"thread_id": "chat1"}}

In [ ]:
# Alternative test data
questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 100/4?",
    "what is 15-7?",
    "what is 3*3?",
    "what is 4*4?"
]

for q in questions:
    response=agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='8bc1ab47-ec8b-466e-b84a-84560b0672ba'), AIMessage(content='2 + 2 = 4', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 14, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_80e729739d', 'id': 'chatcmpl-DJjO5ZShQYE9fcjU7IRllx2jhXeWj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cf281-18e8-7662-8199-843371b91622-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 7, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 

### Summarization based on token limits

In [25]:
from langchain.tools import tool


@tool
def search_hotels(city: str) -> str:
    """search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}: 
    1. grand hotel - 5 star, $350/night, spa, pool, gym,
    2. City Inn - 4 star, $180/night, business center
    3. Budget stay - 3 star, $75/night, free wifi
    """


agent = create_agent(
    model="gpt-4.1",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4.1-mini",
            trigger=("tokens", 550),  # condition based on the tokens
            keep=("tokens", 200),
        ),
    ],
)

config1 = {"configurable": {"thread_id": "chat2"}}

# token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ~ 1 token

In [26]:
# run test

cities = ["paris", "london", "tokyo", "new york", "dubai", "singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find hotels in {city}")]}, config=config1
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

paris: ~129 tokens, 4 messages
[HumanMessage(content='find hotels in paris', additional_kwargs={}, response_metadata={}, id='bd4110eb-6b21-4988-8583-22fcc541a037'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 53, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_decd0450c9', 'id': 'chatcmpl-DJjsupcSsb32ZfvvnLRM6653kxrso', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf29e-4449-7c72-aa7a-3a4e6848f38a-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'paris'}, 'id': 'call_k6CjyJvaCr7nSmcnfiqyWv4v', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'o

### Summarization  based on Fraction(context size)

In [29]:
from langchain.tools import tool


@tool
def search_hotels(city: str) -> str:
    """search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}: 
    1. grand hotel - 5 star, $350/night, spa, pool, gym,
    2. City Inn - 4 star, $180/night, business center
    3. Budget stay - 3 star, $75/night, free wifi
    """


agent = create_agent(
    model="gpt-4.1",    
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4.1-mini",
            trigger=("fraction", 0.005),  # 0.5% =~ 640 tokens 
            keep=("fraction", 0.002), # 0.2% =~ 256 tokens   
        ),
    ],
)

config1 = {"configurable": {"thread_id": "chat2"}}

# token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ~ 1 token

In [30]:
# run test

cities = ["paris", "london", "tokyo", "new york", "dubai", "singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find hotels in {city}")]}, config=config1
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens/128000 # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

paris: ~119 tokens (0.0930%), 4 messages
[HumanMessage(content='find hotels in paris', additional_kwargs={}, response_metadata={}, id='4870a385-3993-4a17-b060-d768a5d63421'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 53, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_decd0450c9', 'id': 'chatcmpl-DJk9D6CFDKUsbmfp5NLurYTchz2la', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf2ad-b01b-7692-ae90-408a7007dbe9-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'paris'}, 'id': 'call_TyPfPHq5D0GF7l6mVWuU8PL7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_token